# 04. Error analysis

Notebook para documentar falsos positivos, falsos negativos y patrones de error.


In [ ]:
## Objetivo del análisis de errores

El propósito de este cuaderno es examinar cualitativamente los errores cometidos por los modelos evaluados, con especial atención a los falsos positivos y falsos negativos. Este análisis permite identificar patrones de fallo, comprender mejor las limitaciones del sistema y proponer líneas de mejora futuras.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = BASE_DIR / "data" / "processed"
OUTPUTS_TABLES = BASE_DIR / "outputs" / "tables"
OUTPUTS_FIGURES = BASE_DIR / "outputs" / "figures"
OUTPUTS_METRICS = BASE_DIR / "outputs" / "metrics"

OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_METRICS.mkdir(parents=True, exist_ok=True)



In [2]:
import joblib

test_df = pd.read_csv(DATA_PROCESSED / "test.csv")
ext_df = pd.read_csv(DATA_PROCESSED / "external_validation_detoxis.csv")
baseline_model = joblib.load(BASE_DIR / "models" / "artifacts" / "baseline_tfidf_logreg.joblib")


In [3]:
test_df["baseline_pred"] = baseline_model.predict(test_df["text_clean"])
ext_df["baseline_pred"] = baseline_model.predict(ext_df["text_clean"])

test_df["error_type"] = np.where(
    test_df["label"] == test_df["baseline_pred"],
    "correcto",
    np.where(test_df["label"] == 1, "FN", "FP")
)

ext_df["error_type"] = np.where(
    ext_df["label"] == ext_df["baseline_pred"],
    "correcto",
    np.where(ext_df["label"] == 1, "FN", "FP")
)

test_df["error_type"].value_counts(), ext_df["error_type"].value_counts()


(error_type
 correcto    1095
 FP           230
 FN           214
 Name: count, dtype: int64,
 error_type
 correcto    2079
 FP           838
 FN           517
 Name: count, dtype: int64)

In [4]:
errors_test = test_df.loc[test_df["error_type"] != "correcto", ["text_clean","label","baseline_pred","error_type","dataset_source"]].head(50)
errors_ext = ext_df.loc[ext_df["error_type"] != "correcto", ["text_clean","label","baseline_pred","error_type","dataset_source"]].head(50)

errors_test.to_csv(OUTPUTS_TABLES / "baseline_errors_test.csv", index=False, encoding="utf-8-sig")
errors_ext.to_csv(OUTPUTS_TABLES / "baseline_errors_external.csv", index=False, encoding="utf-8-sig")

errors_test.head(10)


,text_clean,label,baseline_pred,error_type,dataset_source
0,"te lo mereces wey, eres un chingón, te la rifa...",0,1,FP,multilingual_toxicity_dataset_es
7,"no seas pendejo, esa oferta es una estafa, te ...",0,1,FP,clandestino
15,"la gente no sabe ni lo que dice, mongoloide es...",1,0,FN,multilingual_toxicity_dataset_es
18,ahora es a los españoles a quienes llaman idio...,1,0,FN,clandestino
20,"¿y el asesino de los niños y los ancianos, el ...",0,1,FP,clandestino
21,la verdad yo no voy hacer como ustedes… no voy...,0,1,FP,multilingual_toxicity_dataset_es
22,me estaban chingue y chingue con eso y yo todo...,0,1,FP,clandestino
23,¡ya me contaréis! un tipo despreciable: parási...,1,0,FN,multilingual_toxicity_dataset_es
24,tu amigo panzón y le dices que tiene más curva...,0,1,FP,clandestino
25,"la lujuriosa hermana mayor de las anarchy, es ...",1,0,FN,clandestino


## Comparación de errores entre baseline y BETO

En esta sección se repite el análisis de errores utilizando el modelo transformer BETO ya entrenado. El objetivo es comparar los errores cometidos por el baseline clásico y por el transformer, identificando en qué casos BETO corrige errores previos y en cuáles persisten las dificultades de clasificación.

El análisis se realizará tanto sobre el conjunto de prueba interno (`test.csv`) como sobre el conjunto de validación externa (`external_validation_detoxis.csv`). De esta forma, será posible observar no solo la mejora global en métricas, sino también los patrones cualitativos en los que el modelo basado en transformadores ofrece ventajas frente al enfoque clásico.


In [ ]:
## Observaciones del análisis de errores

El análisis de errores debe centrarse en aquellos casos en los que el modelo confunde expresiones ofensivas genéricas con toxicidad dirigida, así como en ejemplos donde la toxicidad aparece de forma implícita, irónica o contextual. En particular, conviene revisar comentarios que contienen groserías sin objetivo explícito, referencias ambiguas a colectivos, sarcasmo, ataques indirectos o diferencias léxicas entre dominios de entrenamiento y validación externa.

En el caso del baseline, es esperable encontrar una mayor dependencia de palabras concretas o n-gramas frecuentes, lo que puede producir falsos positivos en textos con vocabulario ofensivo, aunque no exista un contenido realmente tóxico. Por su parte, el transformer, aunque mejora el rendimiento global, puede seguir fallando en ejemplos altamente ambiguos, con contexto incompleto o con referencias culturales específicas.

In [ ]:
## Conclusiones del análisis de errores

El análisis cualitativo de errores constituye un complemento indispensable de las métricas cuantitativas, ya que permite comprender no solo cuánto falla un modelo, sino también por qué lo hace. En este proyecto, los errores más relevantes previsiblemente estarán relacionados con la ambigüedad contextual, la ironía, las groserías no dirigidas y el cambio de dominio entre datasets.

Este análisis resulta especialmente útil para justificar futuras mejoras, tales como el ajuste fino con más datos del dominio objetivo, la construcción de un conjunto propio de YouTube, la revisión de criterios de anotación o la incorporación de técnicas de explicabilidad. De este modo, el error analysis no solo documenta limitaciones, sino que también orienta el desarrollo posterior del sistema.

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

In [6]:
MODEL_DIR = BASE_DIR / "models" / "artifacts" / "beto_toxicity_binary"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
transformer_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
transformer_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [7]:
def predict_transformer(texts, model, tokenizer, batch_size=16, max_length=256):
    preds = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encodings = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            outputs = model(**encodings)
            batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            preds.extend(batch_preds.tolist())
    
    return preds

In [8]:
test_df["beto_pred"] = predict_transformer(
    test_df["text_clean"].astype(str).tolist(),
    transformer_model,
    tokenizer
)

ext_df["beto_pred"] = predict_transformer(
    ext_df["text_clean"].astype(str).tolist(),
    transformer_model,
    tokenizer
)

In [9]:
test_df["beto_error_type"] = np.where(
    test_df["label"] == test_df["beto_pred"],
    "correcto",
    np.where(test_df["label"] == 1, "FN", "FP")
)

ext_df["beto_error_type"] = np.where(
    ext_df["label"] == ext_df["beto_pred"],
    "correcto",
    np.where(ext_df["label"] == 1, "FN", "FP")
)

display(test_df["beto_error_type"].value_counts())
display(ext_df["beto_error_type"].value_counts())

beto_error_type
correcto    1192
FP           187
FN           160
Name: count, dtype: int64

beto_error_type
correcto    2472
FN           672
FP           290
Name: count, dtype: int64

In [12]:
if "error_type" in test_df.columns and "baseline_error_type" not in test_df.columns:
    test_df["baseline_error_type"] = test_df["error_type"]

if "error_type" in ext_df.columns and "baseline_error_type" not in ext_df.columns:
    ext_df["baseline_error_type"] = ext_df["error_type"]

print(test_df.columns.tolist())
print(ext_df.columns.tolist())

['text', 'text_clean', 'label', 'dataset_source', 'baseline_pred', 'error_type', 'beto_pred', 'beto_error_type', 'baseline_error_type']
['text', 'text_clean', 'label', 'dataset_source', 'baseline_pred', 'error_type', 'beto_pred', 'beto_error_type', 'baseline_error_type']


In [13]:
comparison_test = pd.DataFrame({
    "baseline": test_df["baseline_error_type"].value_counts(),
    "beto": test_df["beto_error_type"].value_counts()
}).fillna(0).astype(int)

comparison_ext = pd.DataFrame({
    "baseline": ext_df["baseline_error_type"].value_counts(),
    "beto": ext_df["beto_error_type"].value_counts()
}).fillna(0).astype(int)

print("Comparación en TEST")
display(comparison_test)

print("Comparación en VALIDACIÓN EXTERNA")
display(comparison_ext)

comparison_test.to_csv(OUTPUTS_TABLES / "error_comparison_test.csv", encoding="utf-8-sig")
comparison_ext.to_csv(OUTPUTS_TABLES / "error_comparison_external.csv", encoding="utf-8-sig")

Comparación en TEST


,baseline,beto
correcto,1095,1192
FP,230,187
FN,214,160


Comparación en VALIDACIÓN EXTERNA


,baseline,beto
FN,517,672
FP,838,290
correcto,2079,2472


In [14]:
test_improved = test_df[
    (test_df["baseline_error_type"] != "correcto") &
    (test_df["beto_error_type"] == "correcto")
][["text_clean", "label", "baseline_pred", "beto_pred", "dataset_source"]].head(30)

ext_improved = ext_df[
    (ext_df["baseline_error_type"] != "correcto") &
    (ext_df["beto_error_type"] == "correcto")
][["text_clean", "label", "baseline_pred", "beto_pred", "dataset_source"]].head(30)

print("Ejemplos en TEST donde BETO corrigió al baseline")
display(test_improved)

print("Ejemplos en EXTERNAL donde BETO corrigió al baseline")
display(ext_improved)

test_improved.to_csv(OUTPUTS_TABLES / "beto_improves_over_baseline_test.csv", index=False, encoding="utf-8-sig")
ext_improved.to_csv(OUTPUTS_TABLES / "beto_improves_over_baseline_external.csv", index=False, encoding="utf-8-sig")

Ejemplos en TEST donde BETO corrigió al baseline


,text_clean,label,baseline_pred,beto_pred,dataset_source
15,"la gente no sabe ni lo que dice, mongoloide es...",1,0,1,multilingual_toxicity_dataset_es
20,"¿y el asesino de los niños y los ancianos, el ...",0,1,0,clandestino
21,la verdad yo no voy hacer como ustedes… no voy...,0,1,0,multilingual_toxicity_dataset_es
22,me estaban chingue y chingue con eso y yo todo...,0,1,0,clandestino
23,¡ya me contaréis! un tipo despreciable: parási...,1,0,1,multilingual_toxicity_dataset_es
24,tu amigo panzón y le dices que tiene más curva...,0,1,0,clandestino
25,"la lujuriosa hermana mayor de las anarchy, es ...",1,0,1,clandestino
27,estar entretenidos con el demente cristiano lo...,1,0,1,multilingual_toxicity_dataset_es
32,los que toman el super internacional arroz con...,0,1,0,multilingual_toxicity_dataset_es
39,pues mira si te voy contestar no querer el f-3...,0,1,0,clandestino


Ejemplos en EXTERNAL donde BETO corrigió al baseline


,text_clean,label,baseline_pred,beto_pred,dataset_source
2,"asi me gusta, que se maten entre ellos y en al...",1,0,1,detoxis
24,"y todos esos, incluido el loco ese, en malaga....",0,1,0,detoxis
26,vienen los mejores (sicarios) y los mas valien...,1,0,1,detoxis
53,jajaja acojámosles,0,1,0,detoxis
65,en ronda no tenéis nada para él? le queda mas ...,0,1,0,detoxis
74,se que estás de coña pero emigrante es el que ...,0,1,0,detoxis
92,que el karma te oiga.,0,1,0,detoxis
106,estaría bien que se inmolasen en las propias p...,1,0,1,detoxis
117,"detenido pero en tierras españolas, que es lo ...",0,1,0,detoxis
121,"los mejores, los mas degolladores",1,0,1,detoxis


In [15]:
test_improved = test_df[
    (test_df["baseline_error_type"] != "correcto") &
    (test_df["beto_error_type"] == "correcto")
][["text_clean", "label", "baseline_pred", "beto_pred", "dataset_source"]].head(30)

ext_improved = ext_df[
    (ext_df["baseline_error_type"] != "correcto") &
    (ext_df["beto_error_type"] == "correcto")
][["text_clean", "label", "baseline_pred", "beto_pred", "dataset_source"]].head(30)

print("Ejemplos en TEST donde BETO corrigió al baseline")
display(test_improved)

print("Ejemplos en EXTERNAL donde BETO corrigió al baseline")
display(ext_improved)

test_improved.to_csv(OUTPUTS_TABLES / "beto_improves_over_baseline_test.csv", index=False, encoding="utf-8-sig")
ext_improved.to_csv(OUTPUTS_TABLES / "beto_improves_over_baseline_external.csv", index=False, encoding="utf-8-sig")

Ejemplos en TEST donde BETO corrigió al baseline


,text_clean,label,baseline_pred,beto_pred,dataset_source
15,"la gente no sabe ni lo que dice, mongoloide es...",1,0,1,multilingual_toxicity_dataset_es
20,"¿y el asesino de los niños y los ancianos, el ...",0,1,0,clandestino
21,la verdad yo no voy hacer como ustedes… no voy...,0,1,0,multilingual_toxicity_dataset_es
22,me estaban chingue y chingue con eso y yo todo...,0,1,0,clandestino
23,¡ya me contaréis! un tipo despreciable: parási...,1,0,1,multilingual_toxicity_dataset_es
24,tu amigo panzón y le dices que tiene más curva...,0,1,0,clandestino
25,"la lujuriosa hermana mayor de las anarchy, es ...",1,0,1,clandestino
27,estar entretenidos con el demente cristiano lo...,1,0,1,multilingual_toxicity_dataset_es
32,los que toman el super internacional arroz con...,0,1,0,multilingual_toxicity_dataset_es
39,pues mira si te voy contestar no querer el f-3...,0,1,0,clandestino


Ejemplos en EXTERNAL donde BETO corrigió al baseline


,text_clean,label,baseline_pred,beto_pred,dataset_source
2,"asi me gusta, que se maten entre ellos y en al...",1,0,1,detoxis
24,"y todos esos, incluido el loco ese, en malaga....",0,1,0,detoxis
26,vienen los mejores (sicarios) y los mas valien...,1,0,1,detoxis
53,jajaja acojámosles,0,1,0,detoxis
65,en ronda no tenéis nada para él? le queda mas ...,0,1,0,detoxis
74,se que estás de coña pero emigrante es el que ...,0,1,0,detoxis
92,que el karma te oiga.,0,1,0,detoxis
106,estaría bien que se inmolasen en las propias p...,1,0,1,detoxis
117,"detenido pero en tierras españolas, que es lo ...",0,1,0,detoxis
121,"los mejores, los mas degolladores",1,0,1,detoxis


In [ ]:
## Conclusiones del análisis comparativo de errores

La comparación entre el baseline y BETO permite observar no solo una mejora cuantitativa en términos de Macro-F1, sino también una mejora cualitativa en la capacidad de clasificación. En general, BETO corrige parte de los errores cometidos por el baseline, especialmente en ejemplos donde el contexto semántico resulta relevante o donde las expresiones tóxicas no dependen únicamente de palabras aisladas.

No obstante, persisten errores en ambos modelos, sobre todo en casos de ambigüedad contextual, ironía, groserías no dirigidas y comentarios cuya toxicidad depende de información pragmática adicional. Estos resultados confirman que el uso de transformadores mejora el rendimiento del sistema, aunque no elimina por completo las dificultades inherentes a la detección automática de toxicidad en lenguaje natural.